# Notebook Metadata Bootstrap Example

This notebook demonstrates how to use the `dd_cleaner.notebook_utils` APIs to initialize a notebook session, discover available artifacts, and expose dataset bootstrap metadata through the metadata authority table.


In [4]:
import sys
from pathlib import Path

config_file_name = 'config.yaml'

# Ensure we import from the local repository source tree, not an installed package.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = next((p for p in candidates if (p / config_file_name).exists()), None)
if repo_root is None:
    raise FileNotFoundError(f'Could not locate {config_file_name} in the expected repository paths.')

sys.path.insert(0, str(repo_root / 'src'))
from dd_cleaner.notebook_utils import init_notebook_session, get_metadata_table, get_dataset_metadata

working_dir = repo_root
config_path = repo_root / config_file_name

print('Using workspace:', working_dir)
print('Using config:', config_path)


Using workspace: /home/rajiv/programming/dd-parser-cleaner/tests
Using config: /home/rajiv/programming/dd-parser-cleaner/tests/config.yaml


In [5]:
coord, artifacts = init_notebook_session(str(working_dir), config_path=str(config_path))

print('Available artifacts:')
display(artifacts)


✅ Notebook session initialized for workspace: /home/rajiv/programming/dd-parser-cleaner/tests

Available Artifacts:

Artifact Name                                  File Name  \
0                         Raw Data                          sba_loans_raw.csv   
1                     Cleaned Data                    sba_loans_raw_clean.csv   
2                User Cleaned Data             sba_loans_raw_user_cleaned.csv   
3             Tagged Entities (DD)         sba_loans_raw_analysis_results.csv   
4  Cleaning Recommendations Report                cleaning_recommendations.md   
5                 Profiling Report          sba_loans_raw_profiling_report.md   
6                   Handshake File  sba_loans_raw_parser_cleaner_handshake.md   
7                  Quarantine File               sba_loans_raw_quarantine.csv   
8               Metadata Authority           sba_loans_raw_metadata_table.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2     data/dd_cleaner/sba_loans_raw_user_cleaned.csv    True  
3  documents/dd_analysis_results/sba_loans_raw_an...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
6  documents/dd_cleaner/sba_loans_raw_parser_clea...    True  
7       data/quarantine/sba_loans_raw_quarantine.csv   False  
8   data/dd_cleaner/sba_loans_raw_metadata_table.csv   False

Available artifacts:


,Artifact Name,File Name,Location,Exists
0,Raw Data,sba_loans_raw.csv,data/sba_loans_raw.csv,True
1,Cleaned Data,sba_loans_raw_clean.csv,data/dd_cleaner/sba_loans_raw_clean.csv,True
2,User Cleaned Data,sba_loans_raw_user_cleaned.csv,data/dd_cleaner/sba_loans_raw_user_cleaned.csv,True
3,Tagged Entities (DD),sba_loans_raw_analysis_results.csv,documents/dd_analysis_results/sba_loans_raw_an...,True
4,Cleaning Recommendations Report,cleaning_recommendations.md,documents/dd_cleaner/cleaning_recommendations.md,True
5,Profiling Report,sba_loans_raw_profiling_report.md,documents/dd_cleaner/sba_loans_raw_profiling_r...,True
6,Handshake File,sba_loans_raw_parser_cleaner_handshake.md,documents/dd_cleaner/sba_loans_raw_parser_clea...,True
7,Quarantine File,sba_loans_raw_quarantine.csv,data/quarantine/sba_loans_raw_quarantine.csv,False
8,Metadata Authority,sba_loans_raw_metadata_table.csv,data/dd_cleaner/sba_loans_raw_metadata_table.csv,False


In [6]:
if coord.synchronized_dictionary_path.exists():
    df_metadata = get_metadata_table(coord)
    dataset_metadata = get_dataset_metadata(coord)

    print('Dataset-level bootstrap metadata (separate artifact):')
    display(dataset_metadata)

    print('\nPer-attribute metadata authority table (row-level):')
    print(list(df_metadata.columns))
    display(df_metadata.head())
else:
    print('The cleaner baseline has not been established yet.')
    print('Please run the cleaner pipeline before calling get_metadata_table().')
    print('Example command:')
    print(f'  uv run clean-dataset --config {config_path} --action full')
    print('Then rerun this cell.')


Dataset-level bootstrap metadata (separate artifact):


{'dataset_type': 'cross-sectional',
 'subject': 'loan',
 'subject_id_attribute': None,
 'wide_short_homogeneous': False,
 'wide_short_representative_column': None,
 'graph_type': None,
 'notes': 'Generated by dataset bootstrapping.',
 'use_case_answers': {'use_case': 'Determine if the loan is going to default',
  'analysis_objective': 'Identify the at risk loans in the portfolio'}}


Per-attribute metadata authority table (row-level):
['Field Name', 'Definition', 'physical_type', 'logical_type', 'attribute_name', 'provisional_entity_assignment', 'static_dynamic', 'dataset_type', 'subject', 'subject_id_attribute', 'wide_short_homogeneous', 'wide_short_representative_column', 'graph_type', 'notes']


,Field Name,Definition,physical_type,logical_type,attribute_name,provisional_entity_assignment,static_dynamic,dataset_type,subject,subject_id_attribute,wide_short_homogeneous,wide_short_representative_column,graph_type,notes
0,asofdate,Date when the data was recorded,datetime,datetime,asofdate,Logical Categories,none,cross-sectional,loan,None,False,None,None,Generated by dataset bootstrapping.
1,program,Indicator of whether loan was approved under S...,int,numeric,program,Logical Categories,none,cross-sectional,loan,None,False,None,None,Generated by dataset bootstrapping.
2,locationid,SBA's unique lender ID code,int,numeric,locationid,Logical Category: Unique Identifier,none,cross-sectional,loan,None,False,None,None,Generated by dataset bootstrapping.
3,borrname,Borrower name,str,text,borrname,Organization,none,cross-sectional,loan,None,False,None,None,Generated by dataset bootstrapping.
4,borrstreet,Borrower street address,str,text,borrstreet,Geographic,none,cross-sectional,loan,None,False,None,None,Generated by dataset bootstrapping.


## Bootstrap metadata fields exposed by the notebook API

The `get_metadata_table()` function bootstraps the authoritative metadata table from the cleaner's synchronized dictionary and enriches it with dataset bootstrap metadata from `config.yaml`, including fields such as: `dataset_type`, `subject`, `wide_short_homogeneous`, `wide_short_representative_column`, and use-case answers.
